In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
import librosa
from tqdm import tqdm
from multiprocess import Pool
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="Vikhrmodels/ToneWebinars", 
    repo_type="dataset", local_dir="./ToneWebinars", allow_patterns="*/*.parquet")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 379 files: 100%|██████████| 379/379 [00:00<00:00, 2499.94it/s]


'/home/ubuntu/ToneWebinars'

In [3]:
files = glob('ToneWebinars/*/*.parquet')
len(files)

379

In [4]:
# df = pd.read_parquet(files[0])
# df

In [5]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['text'].iloc[i].strip()
            if len(t) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = sf.read(io.BytesIO(b))
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}"
            })
        
    return data

In [6]:
data = multiprocessing(files, loop, cores = 10)

100%|██████████| 9/9 [41:19<00:00, 275.48s/it]


In [7]:
len(data)

308348

In [8]:
with open('ToneWebinars.json', 'w') as fopen:
    json.dump(data, fopen)

In [9]:
audio_files = [d['audio_filename'] for d in data]

with open('ToneWebinars-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [10]:
!du -hs ToneWebinars_audio

72G	ToneWebinars_audio


In [11]:
from glob import glob
import os

repository = 'malaysia-ai/Multilingual-TTS'
folder = 'ToneWebinars_audio'
files = glob(f'{folder}/*.mp3')
len(files)

308348

In [13]:
import zipfile
import time
from huggingface_hub import HfFileSystem
from huggingface_hub import HfApi
api = HfApi()

partition_size = 10e+9

In [14]:
def loop(files):
    files, index = files
    current_index = 0
    api = HfApi()
    fs = HfFileSystem()
    total = 0
    temp = []
    for i in tqdm(range(len(files))):
        s = os.stat(files[i]).st_size
        if s + total >= partition_size:
            part_name = f"{folder}-{index}-{current_index}.zip"
                
            with zipfile.ZipFile(part_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
                for f in temp:
                    zipf.write(f, arcname=f)

            while True:
                try:
                    api.upload_file(
                        path_or_fileobj=part_name,
                        path_in_repo=part_name,
                        repo_id=repository,
                        repo_type="dataset",
                    )
                    break
                except:
                    time.sleep(60)

            os.remove(part_name)
            
            current_index += 1
            temp = [files[i]]
            total = s
        else:
            temp.append(files[i])
            total += s
        
    if len(temp):
        part_name = f"{folder}-{index}-{current_index}.zip"

        with zipfile.ZipFile(part_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for f in temp:
                zipf.write(f, arcname=f)

        while True:
            try:
                api.upload_file(
                    path_or_fileobj=part_name,
                    path_in_repo=part_name,
                    repo_id=repository,
                    repo_type="dataset",
                )
                break
            except:
                time.sleep(60)

        os.remove(part_name)

In [17]:
# multiprocessing(files, loop, cores = 10, returned = False)

In [20]:
# !zip -rq ToneWebinars_audio_neucodec.zip ToneWebinars_audio_neucodec

In [21]:
# !hf upload malaysia-ai/Multilingual-TTS ToneWebinars_audio_neucodec.zip --repo-type=dataset

In [22]:
import json

with open('ToneWebinars.json') as fopen:
    rows = json.load(fopen)

mapping = {}
for i in tqdm(range(len(rows))):
    mapping[rows[i]['audio_filename']] = i
len(mapping)

100%|██████████| 308348/308348 [00:00<00:00, 2571485.02it/s]


308348

In [23]:
import faiss
import os
import numpy as np
from tqdm import tqdm

data = {}
d = 192
index = faiss.IndexFlatL2(d)

centroids = []

def assign(x, threshold=0.1):
    if len(centroids) == 0:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return 0
    
    D, I = index.search(np.array([x], dtype=np.float32), 1)
    if D[0][0] > threshold:
        centroids.append(x)
        index.add(np.array([x], dtype=np.float32))
        return len(centroids)-1
    else:
        return I[0][0]
        
for i in tqdm(range(len(rows))):
    index_ = mapping[rows[i]['audio_filename']]
    v_f = f'ToneWebinars_embedding/{index_}.npy'
    if not os.path.exists(v_f):
        continue
    try:
        v = np.load(v_f)
        data[rows[i]['audio_filename']] = assign(v)
    except Exception as e:
        pass

100%|██████████| 308348/308348 [01:22<00:00, 3741.59it/s]


In [24]:
for i in range(len(rows)):
    s = data[rows[i]['audio_filename']]
    rows[i]['speaker'] = rows[i]['speaker'] + f'_{s}'

In [25]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'ToneWebinars_audio/ToneWebinars-data-train-00236-of-00352_0.mp3',
 'text': 'весу вытесненной жидкости. Весу. Именно. Вот сила Архимеда равна весу вытесненной жидкости. Еще раз. Когда мы находимся в свободном падении, не зря это состояние называется невесомость. Там нет веса. Мы летим свободно, и у нас нет веса. Нет веса, нет силы Архимеда. Из-за чего возникает сила Архимеда? Сила Архимеда возникает из-за разности',
 'speaker': 'ToneWebinars_audio_0'}

In [26]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'ToneWebinars')

Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00,  6.85ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):   8%|▊         | 7.02MB / 92.4MB, 35.1MB/s  
Processing Files (0 / 1):  99%|█████████▉| 91.8MB / 92.4MB,  230MB/s  
Processing Files (0 / 1): 100%|█████████▉| 92.3MB / 92.4MB,  115MB/s  
Processing Files (1 / 1): 100%|██████████| 92.4MB / 92.4MB, 77.4MB/s  
Processing Files (1 / 1): 100%|██████████| 92.4MB / 92.4MB, 77.0MB/s  
New Data Upload: 100%|██████████| 92.4MB / 92.4MB, 77.0MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:02<00:00,  2.76s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/71a09271b445785af8c28a35b7eefaadc3597f5f', commit_message='Upload dataset', commit_description='', oid='71a09271b445785af8c28a35b7eefaadc3597f5f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)